# Modelado Predictivo - Recontacto en 7 días
Objetivo: Predecir la variable objetivo `recontacto_7_dias` utilizando diferentes algoritmos y encontrar los factores más determinantes.

In [ ]:
import os
import pyodbc
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# Conexión
connection_string = f"DRIVER={os.getenv('DB_DRIVER', '{ODBC Driver 18 for SQL Server}')};SERVER={os.getenv('DB_SERVER', 'localhost')};DATABASE={os.getenv('DB_NAME', 'ServicioCiudadanoDW')};UID={os.getenv('DB_USER', 'SA')};PWD={os.getenv('DB_PASSWORD', 'J0AhMXX4H0QBrRr8R1U098y8aA1@')};TrustServerCertificate=yes;"
conn = pyodbc.connect(connection_string)
df = pd.read_sql("SELECT * FROM vw_v1_limpio", conn)
conn.close()

# Preparación de datos
X = df.drop(columns=['id_interaccion', 'fecha_contacto', 'fecha_carga', 'recontacto_7_dias'])
y = df['recontacto_7_dias']

# Separar numéricas y categóricas
num_cols = ['duracion_seg', 'espera_seg', 'casos_previos_30d', 'nivel_transferencias']
cat_cols = ['canal', 'motivo_contacto', 'cola_servicio', 'grabacion_autorizada', 'tipo_usuario', 'turno']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Datos preparados.")


In [ ]:
# Modelo 1: Random Forest
rf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("--- Random Forest Report ---")
print(classification_report(y_test, y_pred_rf))
print(f"ROC AUC: {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}")
